# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:

from google.colab import files

uploaded = files.upload()


Saving AviationData.csv to AviationData (1).csv


In [ ]:
import pandas as pd

# Load the uploaded dataset into df
# Using encoding='latin-1' or low_memory=False to handle potential mixed types safely
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Check column names and data types
df.info()

# Count missing values in each column
missing_values = df.isnull().sum().sort_values(ascending=False)
display(missing_values)

# Display summary statistics
display(df.describe(include="all").T)

Dataset shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make       

,0
Schedule,76307
Air.carrier,72241
FAR.Description,56866
Aircraft.Category,56602
Longitude,54516
Latitude,54507
Airport.Code,38757
Airport.Name,36185
Broad.phase.of.flight,27165
Publication.Date,13771


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,WPR23LA045,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Convert the event date to datetime format
df["Event.Date"] = pd.to_datetime(
    df["Event.Date"], errors="coerce"
)

# Standardize the text fields used for filtering
df["Aircraft.Category"] = (
    df["Aircraft.Category"].astype("string").str.strip()
)
df["Amateur.Built"] = (
    df["Amateur.Built"].astype("string").str.strip()
)

# Keep airplane accidents involving non-amateur-built aircraft
# from 1983 onwards
df = df[
    (df["Aircraft.Category"].str.casefold() == "airplane") &
    (df["Amateur.Built"].str.casefold() == "no") &
    (df["Event.Date"] >= pd.Timestamp("1983-01-01"))
].copy()

print("Dataset shape after filtering:", df.shape)
display(df.head())

Dataset shape after filtering: (21447, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,Unknown,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,Unknown,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
### Construct metric for fatal/serious injuries
df["Fatal.Serious.Injuries"] = (
    df["Total.Fatal.Injuries"].fillna(0)
    + df["Total.Serious.Injuries"].fillna(0)
)

df["Fatal.Serious.Injury.Rate"] = np.where(
    df["Total.Occupants"] > 0,
    df["Fatal.Serious.Injuries"] / df["Total.Occupants"],
    np.nan
)

display(
    df[
        [
            "Total.Fatal.Injuries",
            "Total.Serious.Injuries",
            "Fatal.Serious.Injuries",
            "Fatal.Serious.Injury.Rate"
        ]
    ].head()
)

,Total.Fatal.Injuries,Total.Serious.Injuries,Fatal.Serious.Injuries,Fatal.Serious.Injury.Rate
4149,NaN,NaN,0.0,0.0
4150,NaN,NaN,0.0,0.0
4171,1.0,1.0,2.0,1.0
4285,1.0,NaN,1.0,0.2
5957,NaN,NaN,0.0,0.0


In [ ]:
# Convert existing injury and uninjured columns to numeric values
injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

for col in injury_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate Total.Occupants as the sum of all injury types and uninjured passengers
# Filling NaNs with 0 during sum to avoid missing occupant counts where some data exists
df["Total.Occupants"] = (
    df["Total.Fatal.Injuries"].fillna(0) +
    df["Total.Serious.Injuries"].fillna(0) +
    df["Total.Minor.Injuries"].fillna(0) +
    df["Total.Uninjured"].fillna(0)
)

# If all columns were NaN for a row, set the calculated occupants to NaN as well
all_nan = df[injury_columns].isnull().all(axis=1)
df.loc[all_nan, "Total.Occupants"] = np.nan

all_numeric = injury_columns + ["Total.Occupants"]

# Inspect missing values in these columns
display(df[all_numeric].isnull().sum())

# Display summary statistics
display(df[all_numeric].describe())

,0
Total.Fatal.Injuries,2750
Total.Serious.Injuries,2828
Total.Minor.Injuries,2544
Total.Uninjured,711
Total.Occupants,47


,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Total.Occupants
count,18697.000000,18619.000000,18903.000000,20736.000000,21400.000000
mean,0.740868,0.323326,0.230598,7.718364,8.611168
std,6.770638,2.378395,1.753088,35.633119,35.947238
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,1.000000
50%,0.000000,0.000000,0.000000,1.000000,2.000000
75%,0.000000,0.000000,0.000000,2.000000,2.000000
max,295.000000,161.000000,200.000000,588.000000,588.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Standardize the aircraft damage values
df["Aircraft.damage"] = (
    df["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Create an indicator for whether the aircraft was destroyed
df["Aircraft.Destroyed"] = df["Aircraft.damage"].map({
    "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0,
    "None": 0
})

# Check the results
print("Aircraft damage categories:")
display(df["Aircraft.damage"].value_counts(dropna=False))

print("Destroyed indicator:")
display(df["Aircraft.Destroyed"].value_counts(dropna=False))

Aircraft damage categories:


,count
Aircraft.damage,
Substantial,16990
Destroyed,2316
<NA>,1227
Minor,817
Unknown,97


Destroyed indicator:


,count
Aircraft.Destroyed,
0.0,17807
1.0,2316
NaN,1324


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
# Standardize the Make column
df["Make"] = (
    df["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Replace empty strings with missing values
df["Make"] = df["Make"].replace("", pd.NA)

# Count aircraft accidents by manufacturer
print("Number of unique makes:", df["Make"].nunique())

display(df["Make"].value_counts(dropna=False).head(20))

Number of unique makes: 1087


,count
Make,
CESSNA,7146
PIPER,3989
BEECH,1431
BOEING,1264
MOONEY,363
AIRBUS,243
CIRRUS DESIGN CORP,220
AIR TRACTOR INC,219
BELLANCA,219


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Standardize the Model column
df["Model"] = (
    df["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Replace empty model names with missing values
df["Model"] = df["Model"].replace("", pd.NA)

# Check the number of unique models
print("Number of unique models:", df["Model"].nunique())

# Display the 20 most common models
display(df["Model"].value_counts(dropna=False).head(20))

Number of unique models: 3467


,count
Model,
172,772
737,403
152,317
182,304
172S,278
PA28,273
SR22,265
172N,250
180,213


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# Clean Engine.Type
df["Engine.Type"] = (
    df["Engine.Type"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Clean Weather.Condition
df["Weather.Condition"] = (
    df["Weather.Condition"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Clean Purpose.of.flight
df["Purpose.of.flight"] = (
    df["Purpose.of.flight"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Clean Broad.phase.of.flight
df["Broad.phase.of.flight"] = (
    df["Broad.phase.of.flight"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Convert Number.of.Engines to numeric
df["Number.of.Engines"] = pd.to_numeric(
    df["Number.of.Engines"], errors="coerce"
)

# Treat zero or negative engine counts as missing
df.loc[df["Number.of.Engines"] <= 0, "Number.of.Engines"] = np.nan

# Check the cleaned columns
columns_to_check = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
    "Number.of.Engines"
]

for col in columns_to_check:
    print(f"\n{col}:")
    display(df[col].value_counts(dropna=False).head(10))


Engine.Type:


,count
Engine.Type,
Reciprocating,15015
<NA>,3956
Turbo Prop,1251
Turbo Fan,929
Unknown,134
Turbo Jet,133
Geared Turbofan,12
Turbo Shaft,11
Electric,5



Weather.Condition:


,count
Weather.Condition,
Vmc,17089
<NA>,2979
Imc,1069
Unk,310



Purpose.of.flight:


,count
Purpose.of.flight,
Personal,11714
<NA>,3680
Instructional,2753
Aerial Application,935
Business,499
Unknown,378
Positioning,350
Skydiving,164
Aerial Observation,161



Broad.phase.of.flight:


,count
Broad.phase.of.flight,
<NA>,18619
Landing,1255
Takeoff,509
Cruise,266
Approach,234
Maneuvering,169
Taxi,111
Go-Around,89
Descent,73



Number.of.Engines:


,count
Number.of.Engines,
1.0,15742
2.0,3010
NaN,2560
4.0,93
3.0,40
8.0,1
6.0,1


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Count non-missing values in each column
non_missing_counts = df.notna().sum().sort_values()

# Display columns with fewer than 20,000 non-missing values
columns_to_review = non_missing_counts[
    non_missing_counts < 20000
]

print("Columns with fewer than 20,000 non-missing values:")
display(columns_to_review)

print("Current dataset shape:", df.shape)

Columns with fewer than 20,000 non-missing values:


,0
Schedule,2521
Broad.phase.of.flight,2828
Air.carrier,10162
Airport.Code,13983
Airport.Name,14070
Report.Status,16784
Engine.Type,17491
Purpose.of.flight,17767
Weather.Condition,18468
Total.Serious.Injuries,18619


Current dataset shape: (21447, 33)


In [ ]:
# Identify columns with more than 20,000 non-missing values
columns_to_keep = non_missing_counts[
    non_missing_counts >= 20000
].index.tolist()

# Make sure important analysis columns are retained
important_columns = [
    "Make",
    "Model",
    "Aircraft.damage",
    "Aircraft.Destroyed",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Occupants",
    "Fatal.Serious.Injuries",
    "Fatal.Serious.Injury.Rate",
    "Weather.Condition",
    "Broad.phase.of.flight",
    "Number.of.Engines",
    "Engine.Type",
    "Purpose.of.flight"
]

columns_to_keep = list(
    dict.fromkeys(columns_to_keep + [
        col for col in important_columns if col in df.columns
    ])
)

# Keep only the selected columns
df = df[columns_to_keep].copy()

print("Dataset shape after column removal:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())

Dataset shape after column removal: (21447, 25)
Remaining columns:
['Aircraft.Destroyed', 'Aircraft.damage', 'Publication.Date', 'Injury.Severity', 'Total.Uninjured', 'FAR.Description', 'Registration.Number', 'Total.Occupants', 'Model', 'Location', 'Make', 'Country', 'Aircraft.Category', 'Event.Date', 'Accident.Number', 'Investigation.Type', 'Event.Id', 'Amateur.Built', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Weather.Condition', 'Broad.phase.of.flight', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight']


In [ ]:
print("Fatal.Serious.Injuries" in df.columns)
print("Fatal.Serious.Injury.Rate" in df.columns)

True
True


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
df.to_csv("AviationData_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [ ]:
from google.colab import files
files.download("AviationData_Cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>